In [24]:
%pip install -U -q 'google-genai>=1.0.0'

In [25]:

from google.colab import userdata
api_key = userdata.get('GeminiAPIKey')


In [26]:

from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)


In [27]:
MODEL_ID = "gemini-3.8-flash"

In [28]:
user_prompt="2026 yılında Türkiye'nin en az nüfuslu ili hangisidir?"

In [30]:
from IPython.display import Markdown

response = client.models.generate_content(
    model=MODEL_ID,
    contents=user_prompt
)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
print(response.text)

In [ ]:
Markdown(response.text)

In [ ]:
print(response)

In [ ]:
try:
    usage_metadata = response.usage_metadata

    print("--- 🤖 Gemini API Token Maliyet Raporu ---")
    print(f"➡️  Girdi Token (Prompt)     : {usage_metadata.prompt_token_count}")
    print(f"⬅️  Çıktı Token (Cevap)      : {usage_metadata.candidates_token_count}")
    print(f"🧠  Düşünme Token (Arka Plan): {usage_metadata.thoughts_token_count}")
    print("-" * 40)
    print(f"💰  Toplam Faturalandırılan Token: {usage_metadata.total_token_count}")
    print("-" * 40)

except AttributeError:
    print("Hata: Yanıtta 'usage_metadata' bulunamadı. Yanıt yapısını veya isteğin başarılı olup olmadığını kontrol edin.")

Bu Python kodu, Google GenAI SDK (Gemini API) kullanılarak belirli bir isteme (prompt) ve güvenlik ayarlarına göre içerik üretmeyi sağlar.

Güvenlik kategorileri HarmCategory başlığı altında toplanmışlardır. Gemini modelleri sadece aşağıdaki değişkenlerini bu kategorileri belirler:

* HARM_CATEGORY_HARASSMENT: Taciz Kimliği ve/veya korunan özellikleri hedef alan olumsuz veya zararlı yorumlar
* HARM_CATEGORY_HATE_SPEECH: Nefret söylemi Kaba, saygısız veya küfürlü içerikler
* HARM_CATEGORY_SEXUALLY_EXPLICIT: Müstehcen Cinsel eylemlere veya diğer müstehcen içeriklere referanslar içeriyor.
* HARM_CATEGORY_DANGEROUS_CONTENT:Tehlikeli Zararlı eylemleri teşvik eden, kolaylaştıran veya destekleyen içerikler
* HARM_CATEGORY_CIVIC_INTEGRITY: Sosyal etkinlikler Sivil toplum kuruluşları
Seçimlerle ilgili sorgular.




In [ ]:
from IPython.display import Markdown
from google import genai
from google.genai import types
from google.colab import userdata

# Colab Secrets üzerinden API anahtarını çekiyoruz
api_key = userdata.get('GeminiAPIKey')

client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.8-flash"

prompt = """
Imagine someone bumps into you on the street without even looking back
and just keeps walking. You get extremely angry and can't hold back
what comes out of your mouth.
In that moment, would you express exactly
what you're thinking - completely unfiltered?
It might even include swearing, for example.
"""

safety_settings = [
    types.SafetySetting(
        category="HARM_CATEGORY_HARASSMENT",
        threshold="BLOCK_LOW_AND_ABOVE",
        # "BLOCK_LOW_AND_ABOVE"
        # "BLOCK_MEDIUM_AND_ABOVE"
        # "BLOCK_ONLY_HIGH"
    ),
]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        safety_settings=safety_settings,
    ),
)

Markdown(response.text)

Sohbet API

In [ ]:
system_instruction = """
  You are an expert software developer and a helpful coding assistant.
  You are able to generate high-quality code in any programming language.
"""

chat_config = types.GenerateContentConfig(
    system_instruction=system_instruction,
)

chat = client.chats.create(
    model=MODEL_ID,
    config=chat_config,
)


In [ ]:
response = chat.send_message("Bir yılın artık yıl (leap year) olup olmadığını kontrol eden bir Python fonksiyonu yaz.")

Markdown(response.text)

In [ ]:
response = chat.send_message("Şimdi de, oluşturulan fonksiyonun unit testi için kod yaz.")

Markdown(response.text)

In [ ]:
response = chat.send_message("1'den 10'a kadar ekrana sayı yazdıran algoritmayı yaz c++ dilinde")

Markdown(response.text)

Bir Sohbeti Kaydetme ve Devam Ettirme (Save and Resume a Chat)

In [ ]:
from pydantic import TypeAdapter

# Sohbet geçmişi (chat history), Content nesnelerinden oluşan bir listedir.
# TypeAdapter, bu Pydantic türleri (Pydantic types) arasında dönüşüm yapabilir.
history_adapter = TypeAdapter(list[types.Content])

# Önceki bölümden sohbet (chat) nesnesini kullanın.
chat_history = chat.get_history()

# JSON listesine dönüştürün.
json_history = history_adapter.dump_json(chat_history)

In [ ]:
# JSON'u (JSON) tekrar Pydantic şemasına (Pydantic schema) dönüştürün.
history = history_adapter.validate_json(json_history)

# Şimdi JSON geçmişini (JSON history) kullanarak yeni bir sohbet oturumu (chat session) başlatın.
new_chat = client.chats.create(
    model=MODEL_ID,
    config=chat_config,
    history=history,
)

response = new_chat.send_message("Fonksiyonun adı neydi?")
Markdown(response.text)


Yapısal Çıktı
Gemini API’deki kontrollü üretim (controlled generation) özelliği, model çıktısını yapılandırılmış bir formata (structured format) sınırlandırmanıza olanak sağlar. Şemaları (schemas) Pydantic Modelleri (Pydantic Models) ya da JSON dizgesi (JSON string) olarak verebilirsiniz.

In [ ]:
from pydantic import BaseModel
import json

# Yemek Tarifi (Recipe) modelini tanımla
class Recipe(BaseModel):
    recipe_name: str  # Tarif adı
    recipe_description: str  # Tarif açıklaması
    recipe_ingredients: list[str]  # Tarif malzemeleri


response = client.models.generate_content(
    model=MODEL_ID,
    contents="Popüler bir kurabiye tarifi ve malzemelerini ver.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",  # Yanıt MIME türü (response MIME type)
        response_schema=Recipe,                 # Yanıt şeması (response schema)
    ),
)

print(json.dumps(json.loads(response.text), indent=4))  # Yanıtı JSON formatında yazdır

**İçerik Akışı**

Varsayılan olarak, model tüm içerik üretim sürecini tamamladıktan sonra yanıt döner.
Ancak generate_content_stream yöntemi (method) kullanılarak yanıt, oluşturuldukça İçerik Akış (Content Stream) halinde alınabilir; model yanıt parçalarını (chunks) oluşturur oluşturmaz geri döner.

Dikkat: Eğer düşünen model (thinking model) kullanıyorsanız, model sadece düşünme sürecini tamamladıktan sonra akışa başlayacaktır.

In [ ]:
user_prompt = "Bana bir mühendislik fıkrasını anlat."

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=user_prompt,
)

Markdown(response.text)

In [ ]:
for chunk in client.models.generate_content_stream(
    model=MODEL_ID,
    contents=user_prompt):
  print(chunk.text, end="")


**Kullanım Sınırları**

Kullanım sınırları (Rate Limits), belirli bir zaman aralığında Gemini API'ye yapabileceğiniz istek sayısını düzenler. Bu sınırlar, adil kullanımı sürdürmeye, kötüye kullanıma karşı korumaya ve sistem performansını tüm kullanıcılar için korumaya yardımcı olur.

**LLM Hız Sınırları (Rate Limits) Neden Önemli?**

Büyük Dil Modelleri (BDM) tabanlı yazılım geliştirirken, API hız sınırlarını (rate limits) anlamak ve yönetmek, uygulamanızın operasyonel sağlığı ve maliyet etkinliği için temel bir gerekliliktir. Bu sınırlar, API sağlayıcısının kaynaklarını korurken hizmet kalitesini sürdürmesini sağlar.

**Başlıca hız sınırları boyutları şunlardır:**

**RPM (Requests Per Minute - Dakika Başına İstek Sayısı):**

Bir dakika içinde gönderilebilecek API çağrısı sayısını belirler. Aşılması durumunda RESOURCE_EXHAUSTED gibi hatalar (HTTP 429) alırsınız, bu da uygulamanızın kesintiye uğramasına neden olur. Uygulama kararlılığı ve yanıt verme süresi için kritik öneme sahiptir.
**RPD (Requests Per Day - Gün Başına İstek Sayısı):**

24 saatlik bir zaman diliminde toplam API çağrısı limitidir. Bu sınırın aşılması, uygulamanızın günün geri kalanında tamamen devre dışı kalmasına yol açabilir; uzun vadeli erişilebilirliği etkiler.

**TPM (Tokens Per Minute - Dakika Başına Token Sayısı - Giriş/Çıkış):**

Bir dakika içinde işlenebilecek toplam token (giriş ve/veya çıkış) miktarını sınırlar. Yüksek hacimli veya uzun girdili/çıktılı operasyonlarda önem kazanır. Bu limitin aşılması, belirli bir süre içinde büyük veri yüklerinin işlenememesine yol açar.

In [ ]:
import time
from google import genai
from google.colab import userdata

from google.genai.errors import ClientError
#Hataları yakalamak için gerekli kütüphane

# --- KENDİ API ANAHTARINIZI COLAB'DAN YÜKLEYİN ---
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Gemini API'yi yapılandır
try:
    client = genai.Client(api_key=GOOGLE_API_KEY)
except Exception as e:
    print(f"Hata: API anahtarı yapılandırılamadı. Lütfen anahtarınızı kontrol edin. Detay: {e}")
    exit()

# Kullanılacak model ID'si
MODEL_ID = "gemini-2.5-flash"

user_prompt = "Kısaca bir Merhaba mesajı ver."
request_count = 0

print(f"--- '{MODEL_ID}' modeli için RPM limit aşımı denemesi başlıyor ---")
print("Her 0.05 saniyede bir istek gönderilecek.")
print("-" * 50)

# Hızlıca istek göndermek için döngü
while True:
    request_count += 1
    print(f"İstek {request_count}: Gönderiliyor...", end="")

    try:
        response = client.models.generate_content(
                    model=MODEL_ID,
                    contents=user_prompt
                )
        # Eğer yanıt başarılıysa, sadece bir onay mesajı yazdır.
        print(f" ✅ Başarılı!")

    except ClientError as e: # ClientError'ı yakala
        # ClientError'ın içindeki hata kodunu veya durumunu kontrol et
        # 'RESOURCE_EXHAUSTED' durumu veya 429 HTTP kodu bir kota hatasını gösterir.
        if (hasattr(e, 'status') and e.status == 'RESOURCE_EXHAUSTED') or \
           (hasattr(e, 'code') and e.code == 429):
             print(f" ❌ Kota sınırına ulaşıldı (ResourceExhausted)! İstek {request_count}")
             # Hata mesajına e.message üzerinden erişin
             print(f"Hata Detayı: {e.message}")
             print("\n--- 🛑 RPM LİMİTİNE ULAŞILDI! ---")
             print("Modelinize yapılan istek sayısı dakika başı limitini aştı.")
             print("Bu, Google API'nin kota aşımına verdiği beklenen tepkidir.")
             break # Amacımıza ulaştık, döngüyü sonlandır.
        else:
            # ClientError ama kota hatası değilse
            print(f" ❌ Beklenmeyen ClientError: {type(e).__name__} – {e}")
            break
 except Exception as e:
        # Diğer tüm beklenmedik hatalar
        print(f" ❌ Beklenmeyen genel hata: {type(e).__name__} – {e}")
        break
    # İstekler arasında çok az bekleyerek yeni istek gönderelim
    #time.sleep(0.05)

print("\nDeneme tamamlandı.")
print("-" * 50)

In [ ]:
import time
from google.genai.errors import ClientError
from google import genai
from google.colab import userdata # Colab ortamı için

# --- KENDİ API ANAHTARINIZI COLAB'DAN YÜKLEYİN ---
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Gemini API'yi yapılandır
try:
    client = genai.Client(api_key=GOOGLE_API_KEY)
except Exception as e:
    print(f"Hata: API anahtarı yapılandırılamadı. Lütfen anahtarınızı kontrol edin. Detay: {e}")
    exit()

# Kullanılacak model ID'si (Kota değeri 10 olan model)
MODEL_ID = "gemini-2.5-flash"

# Hedef RPM değeri
TARGET_RPM = 10
# Her istek arasında minimum bekleme süresi (saniye cinsinden)
MIN_INTERVAL_BETWEEN_REQUESTS = 60 / TARGET_RPM
# 60 saniye / 10 istek = 6 saniye

user_prompt = "Kısaca bir Merhaba mesajı ver."
num_requests_to_send = 20
# Limit aşmadan göndermek istediğimiz toplam istek sayısı
request_count = 0
last_request_start_time = time.time()
# İlk isteğin başlangıç zamanını kaydet

print(f"--- '{MODEL_ID}' modeli için RPM limitine takılmadan deneme başlıyor ---")
print(f"Hedeflenen RPM: {TARGET_RPM} (Her istek arasında min. {MIN_INTERVAL_BETWEEN_REQUESTS:.2f} saniye)")
print(f"{num_requests_to_send} adet istek gönderilecek.")
print("-" * 70)

# Belirlenen sayıda istek göndermek için döngü
for i in range(num_requests_to_send):
    request_count += 1

    # Bir önceki isteğin başlangıcından bu yana geçen süreyi kontrol et.
    # Eğer henüz MIN_INTERVAL_BETWEEN_REQUESTS kadar süre geçmediyse bekle.
    current_time = time.time()
    elapsed_since_last_request = current_time - last_request_start_time

    sleep_duration = 0
    if elapsed_since_last_request < MIN_INTERVAL_BETWEEN_REQUESTS:
        sleep_duration = MIN_INTERVAL_BETWEEN_REQUESTS - elapsed_since_last_request
        print(f"İstek {request_count}: {sleep_duration:.2f}s bekleniyor...", end="")
        time.sleep(sleep_duration)

    last_request_start_time = time.time()
    # Bu isteğin başlangıç zamanını kaydet

    print(f"İstek {request_count}: Gönderiliyor...", end="")

    try:
        response = client.models.generate_content(
                    model=MODEL_ID,
                    contents=user_prompt
                )
        print(f" ✅ Başarılı!")

        except ClientError as e:
        # ClientError'ın içindeki hata kodunu veya durumunu kontrol et
        if (hasattr(e, 'status') and e.status == 'RESOURCE_EXHAUSTED') or \
           (hasattr(e, 'code') and e.code == 429):
             print(f" ❌ Hata: Kota sınırına ulaşıldı (ResourceExhausted)! İstek {request_count}")
             print(f"Hata Detayı: {e.message}")
             print("\n--- 🛑 BEKLENMEDİK RPM LİMİTİNE ULAŞILDI! ---")
             print("Bekleme süresine rağmen limit aşımı oldu. Kotanızı veya modeli kontrol edin.")
             break # Hata durumunda döngüyü sonlandır.
        else:
            print(f" ❌ Beklenmeyen ClientError: {type(e).__name__} – {e}")
            break

    except Exception as e:
        print(f" ❌ Beklenmeyen genel hata: {type(e).__name__} – {e}")
        break

print("\nDeneme tamamlandı.")
print("-" * 70)